# Clase 09 — Topic modeling: LSA y LDA

**Curso:** NLP Clásico con Python · **Bloque 4:** Significado y búsqueda
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/09_topic_modeling.md`

## Qué vas a ver en esta clase

- La idea de **tema** como una distribución sobre palabras y de **documento** como una mezcla de temas, con un ejemplo que se calcula a mano.
- **LSA**: descomposición en valores singulares truncada (`TruncatedSVD`) sobre TF-IDF, y cómo se leen (y cómo no) sus dimensiones.
- **LDA** con `gensim` (`LdaModel`), que se entrena sobre **recuentos** y no sobre TF-IDF: palabras principales y mezcla de temas de cada documento, con semilla fija.
- Cómo los **nombres propios**, las stopwords y la lematización cambian los temas.
- **Elegir el número de temas** con `CoherenceModel` (c_v y u_mass) y criterio humano, y ver la **inestabilidad** con otras semillas y otros tamaños de documento.
- Comprobar de forma honesta si LSA y LDA **recuperan** los dominios conocidos de las reseñas sintéticas (pureza y ARI).

**Requisitos:** clases 3 y 4 (stopwords, TF-IDF) y 8 (probabilidades; la 10 seguirá con embeddings). **Librerías:** `scikit-learn`, `gensim`, `nltk` (stopwords), `spaCy` (solo para una comparación con lemas), `numpy`, `pandas`, `matplotlib`.
**Datos:** `datos/gutenberg/quijote_cap01-14.txt`, `datos/gutenberg/promessi_sposi_cap01-06.txt`, `datos/resenas_es.csv` y `datos/resenas_it.csv` (en Colab, sube la carpeta `datos/`). Los textos literarios son de dominio público y de hace dos siglos o más: **no representan el español ni el italiano de hoy**. Todas las cifras son un *ejemplo ilustrativo*.

---
## Celda 0 — Setup

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("matplotlib", "matplotlib")
asegurar("scikit-learn", "sklearn")
asegurar("gensim", "gensim")
asegurar("nltk", "nltk")
asegurar("spacy", "spacy")
print("Librerías listas.")

In [ ]:
import os
import re
import math
import random
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans
from sklearn.preprocessing import Normalizer
from sklearn.metrics import adjusted_rand_score
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel

warnings.filterwarnings("ignore")
random.seed(42)                          # semilla fija
np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

def cargar_stopwords(idioma):
    """Lista de stopwords de NLTK (la descarga si falta)."""
    try:
        return set(stopwords.words(idioma))
    except LookupError:
        nltk.download("stopwords", quiet=True)
        return set(stopwords.words(idioma))

def tabla(cabeceras, filas, ancho=14):
    """Imprime una tabla de texto alineada."""
    print("".join(str(c).ljust(ancho) for c in cabeceras))
    print("-" * ancho * len(cabeceras))
    for fila in filas:
        print("".join((f"{x:.3f}" if isinstance(x, float) else str(x)).ljust(ancho) for x in fila))

print("Configuración lista.")

In [ ]:
ENCAB = re.compile(r"^(Capítulo|CAPITOLO)\b")   # dato-es
CORPORA = {"spa": ("gutenberg/quijote_cap01-14.txt", "spanish", "es_core_news_sm"),
           "ita": ("gutenberg/promessi_sposi_cap01-06.txt", "italian", "it_core_news_sm")}

def leer_texto(ruta):
    """Lee un extracto: quita las marcas _cursiva_ y los encabezados de capítulo y devuelve un solo texto."""
    parrafos = [p.strip() for p in open(ruta, encoding="utf-8").read().replace("_", "").split("\n\n") if p.strip()]
    return " ".join(p for p in parrafos if not ENCAB.match(p))

def nombres_propios(texto, minimo=3, umbral=0.8):
    """Heurística: palabras que van con mayúscula al menos el 80 % de las veces que aparecen fuera del inicio de frase."""
    con_mayus, total = Counter(), Counter()
    for m in re.finditer(r"\w+", texto):
        if re.search(r"(^|[.!?…:]\s*[«—\"“]?\s*)$", texto[max(0, m.start() - 4):m.start()]):
            continue                                   # inicio de frase: la mayúscula no informa
        w = m.group()
        total[w.lower()] += 1
        con_mayus[w.lower()] += w[0].isupper()
    return {w for w in total if total[w] >= minimo and con_mayus[w] / total[w] >= umbral}

def trozos(tokens, n=300):
    """Parte una lista de tokens en pseudodocumentos consecutivos de n tokens (el último trozo corto se descarta si tiene menos de n/2)."""
    return [tokens[i:i + n] for i in range(0, len(tokens) - n // 2, n)]

def preparar(tokens, stop, nombres=frozenset(), lemas=None, minimo=3):
    """Minúsculas, fuera stopwords y palabras cortas, fuera nombres propios; si se dan lemas, se usan en lugar de las formas."""
    fuente = lemas if lemas is not None else [w.lower() for w in tokens]
    return [w for w in fuente if w not in stop and len(w) >= minimo and w not in nombres]

def bolsa(docs, no_below=3, no_above=0.25):
    """Diccionario de gensim (filtrado por frecuencia de documento) y recuentos por documento."""
    dic = Dictionary(docs)
    dic.filter_extremes(no_below=no_below, no_above=no_above)
    return dic, [dic.doc2bow(d) for d in docs]

def entrenar_lda(docs, k, semilla=0, no_below=3, no_above=0.25):
    """LDA de gensim sobre RECUENTOS, con semilla fija."""
    dic, bow = bolsa(docs, no_below, no_above)
    m = LdaModel(bow, num_topics=k, id2word=dic, random_state=semilla, passes=10, iterations=50, eval_every=None)
    return m, dic, bow

def temas_lda(m, n=8):
    """Las n palabras más probables de cada tema."""
    return [[w for w, _ in m.show_topic(i, n)] for i in range(m.num_topics)]

def mezcla(m, bow):
    """Matriz documento-tema (cada fila suma 1)."""
    return np.array([[p for _, p in m.get_document_topics(b, minimum_probability=0.0)] for b in bow])

def jaccard_medio(A, B):
    """Para cada tema de A, el mejor Jaccard con un tema de B (sobre sus palabras principales); se promedia."""
    A, B = [set(a) for a in A], [set(b) for b in B]
    return float(np.mean([max(len(a & b) / len(a | b) for b in B) for a in A]))

def coherencia(m, docs, dic, bow, medida="c_v"):
    """Coherencia de gensim (c_v usa una ventana deslizante sobre los textos; u_mass usa los recuentos)."""
    if medida == "u_mass":
        return CoherenceModel(model=m, corpus=bow, dictionary=dic, coherence="u_mass").get_coherence()
    return CoherenceModel(model=m, texts=docs, dictionary=dic, coherence=medida, processes=1).get_coherence()

def pureza(y, grupos):
    """Pureza: fracción de documentos que pertenecen a la clase mayoritaria de su grupo."""
    total = 0
    for g in sorted(set(grupos)):
        total += max(Counter(yi for yi, gi in zip(y, grupos) if gi == g).values())
    return total / len(y)

---
## 1. Tema = distribución sobre palabras; documento = mezcla de temas

Un **tema** es una distribución de probabilidad sobre el vocabulario: `P(palabra | tema)`. Un **documento** es una **mezcla** de temas, con proporciones `θ` que suman 1. La probabilidad de que una palabra del documento sea `w` es

$$P(w\mid d)=\sum_{t} \theta_{d,t}\,P(w\mid t)$$

Ejemplo pequeño (inventado, *ejemplo ilustrativo*): dos temas sobre cuatro palabras. Un documento con `θ = (0,7 ; 0,3)`. Calcula a mano `P(gol | d)` antes de ejecutar (0,7·0,5 + 0,3·0,1).

In [ ]:
# datos-es
# Dos temas inventados sobre un vocabulario de 4 palabras (una fila por tema; cada fila suma 1)
VOCAB = ["gol", "liga", "sal", "receta"]      # en italiano: gol, campionato, sale, ricetta
PHI = np.array([[0.5, 0.3, 0.1, 0.1],          # tema 1 (P(palabra | tema 1))
                [0.1, 0.1, 0.4, 0.4]])         # tema 2

In [ ]:
theta = np.array([0.7, 0.3])
p_doc = theta @ PHI                        # mezcla: P(w | d) = Σ_t θ_t · P(w | t)
for w, p in zip(VOCAB, p_doc):
    print(f"P({w} | d) = {p:.2f}")
print("suma:", p_doc.sum())
assert np.isclose(p_doc[0], 0.7 * 0.5 + 0.3 * 0.1) and np.isclose(p_doc.sum(), 1.0)   # coincide con el cálculo a mano: 0,38

### ¿Qué observas?

`P(gol | d) = 0,38`: aparece mucho en el tema 1 (0,5), poco en el 2 (0,1) y el documento es sobre todo del tema 1. La mezcla es siempre una distribución (suma 1). **LDA hace el camino inverso**: dado el texto, busca `PHI` y `θ` que lo expliquen. Nada en el modelo dice qué "significa" cada tema: solo hay números.

---
## 2. Datos: pseudodocumentos literarios y nombres propios

Un capítulo entero es **un** documento: muy pocos para un modelo de temas. Cortamos cada obra en **trozos consecutivos de 300 tokens** (pseudodocumentos). Decisiones que declaramos porque cambian los temas:

- **Trozos de longitud fija**, sin respetar párrafos ni capítulos (un tema puede quedar partido). Se descarta el último trozo si es muy corto.
- **Nombres propios** (`Quijote`, `Sancho`, `Renzo`, `Lucia`…): dominan cualquier tema. Los detectamos con una **heurística** (palabras que casi siempre van con mayúscula fuera del inicio de frase); tiene falsos positivos (`ah`, `oh`, `Dios`) y no es perfecta.
- **Stopwords** de NLTK, palabras de menos de 3 letras y **filtro de frecuencia de documento** (fuera las palabras que aparecen en menos de 3 trozos o en más del 25 %).

In [ ]:
DATOS = {}
for idioma, (ruta, nombre_nltk, modelo_spacy) in CORPORA.items():
    texto = leer_texto(cargar_datos(ruta))
    tokens = re.findall(r"\w+", texto)
    DATOS[idioma] = dict(tokens=tokens, docs=trozos(tokens, 300), nombres=nombres_propios(texto), stop=cargar_stopwords(nombre_nltk))
    d = DATOS[idioma]
    mas_frecuentes = [w for w, _ in Counter(t.lower() for t in tokens if t.lower() in d["nombres"]).most_common(6)]
    print(idioma, "| tokens:", len(tokens), "| pseudodocumentos:", len(d["docs"]),
          "| nombres detectados:", len(d["nombres"]), "|", mas_frecuentes)
    assert len(d["docs"]) > 90 and all(len(x) == 300 for x in d["docs"])

### ¿Qué observas?

Salen algo más de 100 pseudodocumentos por obra: **decenas**, no miles. Es poco para un modelo probabilístico (lo comprobaremos con la estabilidad). Los nombres más frecuentes son los de los protagonistas, justo lo que va a "secuestrar" los temas.

In [ ]:
# Preparación de referencia: minúsculas, sin stopwords ni palabras cortas, sin nombres propios
for idioma, d in DATOS.items():
    d["base"] = [preparar(x, d["stop"], d["nombres"]) for x in d["docs"]]
    print(idioma, "tokens por documento tras preparar (media):", round(np.mean([len(x) for x in d["base"]]), 1))
    print("  ", d["base"][0][:12])

---
## 3. LSA: TF-IDF + SVD truncada

**LSA** (*Latent Semantic Analysis*) aplica una **descomposición en valores singulares truncada** a la matriz documento-término TF-IDF: `X ≈ U Σ Vᵀ` con solo `k` dimensiones. Cada fila de `Vᵀ` es una **dimensión latente**: un vector de pesos sobre las palabras, que pueden ser **positivos o negativos**. No es una distribución de probabilidad: **no es un "tema" en el sentido de LDA**.

Tres cosas que hay que saber al leerlas:

1. **El signo es arbitrario**: multiplicar una dimensión por −1 da la misma descomposición. Se interpreta el contraste entre palabras de signo opuesto, no el signo.
2. La primera dimensión suele recoger **el nivel general de las palabras** (pesos casi todos del mismo signo) y no un tema.
3. Las demás son **combinaciones ortogonales** que separan los documentos: pueden mezclar asuntos distintos.

In [ ]:
def lsa(docs, k=5, semilla=0, **args):
    """TF-IDF sobre documentos ya tokenizados y SVD truncada; devuelve (svd, vocabulario, matriz de documentos)."""
    tf = TfidfVectorizer(analyzer=lambda d: d, min_df=3, max_df=0.25)
    X = tf.fit_transform(docs)
    svd = TruncatedSVD(n_components=k, random_state=semilla, **args).fit(X)
    return svd, np.array(tf.get_feature_names_out()), X

def extremos(svd, vocab, i, n=6):
    """Palabras con mayor peso positivo y negativo de la dimensión i."""
    comp = svd.components_[i]
    orden = np.argsort(comp)
    return list(vocab[orden[::-1][:n]]), list(vocab[orden[:n]])

for idioma, d in DATOS.items():
    svd, vocab, X = lsa(d["base"])
    d["lsa"] = (svd, vocab, X)
    print("=====", idioma, "| matriz TF-IDF:", X.shape, "| valores singulares:", np.round(svd.singular_values_, 2))
    print("  fracción de pesos positivos por dimensión:", np.round((svd.components_ > 0).mean(axis=1), 2))
    for i in range(4):
        pos, neg = extremos(svd, vocab, i)
        print(f"  dimensión {i} (+)", pos)
        print("      (-)", neg)
    assert np.allclose(svd.components_ @ svd.components_.T, np.eye(5), atol=1e-6)   # las dimensiones son ortonormales

### ¿Qué observas?

- La dimensión 0 tiene el valor singular más grande y, en las demás, los pesos se reparten entre positivos y negativos (mira la fracción de pesos positivos): la primera recoge sobre todo el "nivel general" de las palabras, no un asunto concreto.
- En las demás dimensiones aparecen **polos**: un grupo de palabras positivas frente a otro de negativas. Léelas como un **contraste** (por ejemplo, palabras de un episodio frente a otro), no como un tema con etiqueta.
- **Las etiquetas las pone la persona**: el modelo solo entrega números. Cualquier nombre que les des es una interpretación tuya, discutible.

### El signo es arbitrario (y la solución no es única con algoritmos aleatorios)

Ajustamos la misma SVD con dos semillas y con el algoritmo exacto (`arpack`) y comparamos las dimensiones con su **correlación**: un valor cercano a −1 significa "la misma dimensión con el signo invertido".

In [ ]:
for idioma, d in DATOS.items():
    svd0, vocab, X = d["lsa"]
    svd1 = TruncatedSVD(n_components=5, random_state=1).fit(X)
    svd2 = TruncatedSVD(n_components=5, algorithm="arpack").fit(X)
    corr = lambda a, b: [round(float(np.corrcoef(a.components_[i], b.components_[i])[0, 1]), 3) for i in range(5)]
    print(idioma, "| semilla 0 frente a semilla 1:", corr(svd0, svd1), "| semilla 0 frente a arpack:", corr(svd0, svd2))
    assert abs(corr(svd0, svd1)[0]) > 0.9                     # la dimensión 0 es estable salvo el signo

### ¿Qué observas?

Las primeras dimensiones coinciden (correlación cercana a ±1), las últimas menos, y alguna puede aparecer con **signo negativo**. Conclusión práctica: fija `random_state`, no atribuyas significado al signo y desconfía de las dimensiones de orden alto (recogen cada vez menos varianza).

Ahora los pseudodocumentos en las dimensiones 1 y 2, coloreados por su **posición en el libro** (de 0, inicio, a 1, final).

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for eje, (idioma, d) in zip(ejes, DATOS.items()):
    svd, vocab, X = d["lsa"]
    Z = svd.transform(X)
    pos = np.linspace(0, 1, len(Z))
    graf = eje.scatter(Z[:, 1], Z[:, 2], c=pos, cmap="viridis", s=25)
    eje.set_title(f"LSA — {idioma}")
    eje.set_xlabel("dimensión 1")
    eje.set_ylabel("dimensión 2")
fig.colorbar(graf, ax=ejes, label="posición en el libro")
plt.show()

---
## 4. LDA con gensim: recuentos, semilla y mezcla de temas

**LDA** (*Latent Dirichlet Allocation*, Blei, Ng y Jordan, 2003) es un modelo **generativo**: cada tema es una distribución sobre palabras y cada documento una mezcla de temas; el entrenamiento busca los parámetros que mejor explican los recuentos. Por eso se entrena sobre **recuentos de palabras** (bolsa de palabras, `doc2bow`) y **no sobre TF-IDF** (los pesos TF-IDF no son recuentos y el modelo probabilístico deja de tener sentido).

Con `gensim`: `Dictionary` (vocabulario y filtro) → `doc2bow` → `LdaModel(num_topics=k, random_state=…)`. Fijamos `random_state`, 10 pasadas (`passes`) y 50 iteraciones internas. El **número de temas `k` lo eliges tú**.

In [ ]:
K = 5
for idioma, d in DATOS.items():
    d["lda"] = entrenar_lda(d["base"], K, semilla=0)
    m, dic, bow = d["lda"]
    print("=====", idioma, "| vocabulario:", len(dic), "| documentos:", len(bow))
    for i, palabras in enumerate(temas_lda(m, 8)):
        print(f"  tema {i}:", " ".join(palabras))
    theta = mezcla(m, bow)
    assert theta.shape == (len(bow), K) and np.allclose(theta.sum(axis=1), 1.0, atol=1e-5)     # cada fila es una distribución
    d["theta"] = theta
    print("  peso medio del tema dominante:", round(float(theta.max(axis=1).mean()), 3), "| documentos con tema dominante > 0,9:", int((theta.max(axis=1) > 0.9).sum()), "/", len(theta))
    for j in (0, 40, 80):
        print(f"  documento {j}:", np.round(theta[j], 2), "→ tema dominante", int(theta[j].argmax()))

### ¿Qué observas?

- Cada tema es una lista de palabras con probabilidad; **no trae nombre**. Léelo y decide tú si es coherente (por ejemplo, "una venta, un ventero, unas armas" podría llamarse "escenas de la venta"; otra persona podría llamarlo de otra forma). Otros temas son mezclas sin un hilo claro: con pseudodocumentos cortados a ciegas es normal.
- Cada documento tiene una **mezcla** (`θ`): la fila suma 1 y el "tema dominante" es solo su máximo. Aquí casi todas las mezclas están muy concentradas en un tema (con ~140 palabras por documento y los parámetros por defecto de `gensim`); con otros documentos o con `alpha` distinto serían más repartidas.
- Lo que ves depende de `k`, de la semilla, del filtro y del pre-tratamiento: lo comprobamos ahora.

---
## 5. Experimento guiado: el efecto de los nombres propios

Entrenamos LDA con los mismos parámetros pero **sin quitar los nombres propios** y contamos cuántos de ellos hay entre las 10 palabras principales de los 5 temas (50 posiciones en total).

In [ ]:
def nombres_en_top(m, nombres, n=10):
    return sum(w in nombres for palabras in temas_lda(m, n) for w in palabras)

for idioma, d in DATOS.items():
    d["con_nombres"] = [preparar(x, d["stop"]) for x in d["docs"]]
    m_con = entrenar_lda(d["con_nombres"], K, semilla=0)[0]
    m_sin = d["lda"][0]
    print(idioma, "| nombres en el top-10 con nombres:", nombres_en_top(m_con, d["nombres"]),
          "| sin nombres:", nombres_en_top(m_sin, d["nombres"]))
    for i, palabras in enumerate(temas_lda(m_con, 8)):
        print(f"  tema {i}:", " ".join(palabras))
    assert nombres_en_top(m_sin, d["nombres"]) == 0

### ¿Qué observas?

Con los nombres presentes, buena parte de las palabras principales son nombres de personajes y lugares: los "temas" pasan a ser **quién aparece** (escena del cura, escena de un personaje) más que **de qué se habla**. Quitarlos no es "lo correcto": es una **decisión** que depende de la pregunta (si quieres separar episodios por personajes, conviene dejarlos). Lo importante es declararla.

---
## 6. Experimento guiado: el pre-tratamiento cambia los temas

Comparamos cuatro variantes (LDA, `k = 5`, semilla 0) con la de referencia. Como el propio modelo cambia con la semilla, incluimos una **referencia de ruido**: la misma preparación con la semilla 1.

- `base`: la de arriba (minúsculas, sin stopwords ni nombres, filtro de frecuencia).
- `sin_stop`: no se quitan stopwords ni palabras cortas, y **tampoco** se filtran las palabras muy frecuentes.
- `con_nombres`: los nombres propios se dejan.
- `lemas`: se usan lemas de spaCy (`es_core_news_sm` / `it_core_news_sm`) en lugar de formas.

`Jaccard` mide cuánto se parecen los temas de la variante a los de la referencia (1 = iguales). La **coherencia c_v de variantes con vocabularios distintos no se puede comparar**: la incluimos para mostrar por qué (y para ver cuánto varía con la semilla).

In [ ]:
import subprocess, sys, spacy

def cargar_modelo(nombre):
    """Carga un modelo de spaCy y lo descarga si falta (idempotente)."""
    try:
        return spacy.load(nombre, disable=["parser", "ner"])
    except OSError:
        subprocess.run([sys.executable, "-m", "spacy", "download", "-q", nombre], check=True)
        return spacy.load(nombre, disable=["parser", "ner"])

t0 = time.time()
for idioma, (ruta, nombre_nltk, modelo_spacy) in CORPORA.items():
    d = DATOS[idioma]
    nlp = cargar_modelo(modelo_spacy)
    lemas = [[t.lemma_.lower() for t in doc if t.is_alpha] for doc in nlp.pipe([" ".join(x) for x in d["docs"]])]
    d["variantes"] = {
        "base": d["base"],
        "sin_stop": [preparar(x, set(), d["nombres"], minimo=1) for x in d["docs"]],
        "con_nombres": d["con_nombres"],
        "lemas": [preparar(x, d["stop"], d["nombres"], lemas=l) for x, l in zip(d["docs"], lemas)],
    }
print("lematización hecha")

In [ ]:
def ajustar_variante(nombre, docs, semilla=0):
    no_above = 1.0 if nombre == "sin_stop" else 0.25        # en sin_stop no se filtran las frecuentes
    return entrenar_lda(docs, K, semilla=semilla, no_above=no_above)

for idioma, d in DATOS.items():
    print("=====", idioma)
    referencia = temas_lda(d["lda"][0], 10)
    filas = []
    modelos = {}
    for nombre, docs in list(d["variantes"].items()) + [("base_semilla1", d["base"])]:
        semilla = 1 if nombre == "base_semilla1" else 0
        m, dic, bow = ajustar_variante(nombre.replace("_semilla1", ""), docs, semilla)
        modelos[nombre] = m
        top = temas_lda(m, 10)
        filas.append([nombre, len(dic), round(jaccard_medio(referencia, top), 2),
                      sum(w in d["nombres"] for t in top for w in t), sum(w in d["stop"] for t in top for w in t),
                      round(coherencia(m, docs, dic, bow, "c_v"), 3)])
    tabla(["variante", "|V|", "Jaccard", "nombres", "stopwords", "c_v"], filas, ancho=15)
    for nombre in ("sin_stop", "con_nombres", "lemas"):
        print(f"  {nombre:12s} tema 0:", " ".join(temas_lda(modelos[nombre], 8)[0]))
    d["jac_ruido"] = filas[-1][2]

### ¿Qué observas?

- `sin_stop`: los temas se llenan de `de`, `que`, `la`… (50 de 50 posiciones son stopwords): sin filtro, LDA modela la gramática, no los asuntos.
- `con_nombres`: los temas incluyen nombres. Su c_v no es claramente más bajo que el de `base`, aunque los temas sean menos informativos sobre el asunto: **una coherencia alta no garantiza temas útiles**.
- `lemas`: los temas cambian (Jaccard bajo), aunque **el ruido de la semilla también es grande** (fila `base_semilla1`): con tan poco texto no se puede afirmar que una variante sea "mejor", solo que **el pre-tratamiento y la semilla mueven los temas**.
- Fíjate en el c_v de `base` frente a `base_semilla1`: **la misma preparación, con otra semilla, cambia el c_v tanto como cualquier variante**. **No compares** los c_v de las filas ni entre idiomas: cada fila tiene un vocabulario y un corpus distintos.

---
## 7. Elegir el número de temas: coherencia y criterio humano

La **coherencia** puntúa un tema por la frecuencia con que co-ocurren sus palabras principales. Usamos dos medidas de `gensim`, y hay que **declarar cuál**:

- **c_v**: ventana deslizante sobre los textos y similitud del coseno entre vectores de información mutua normalizada. Valores típicos entre 0 y 1.
- **u_mass**: se calcula con los recuentos del corpus; es negativa (más cerca de 0 = más coherente).

Ninguna es "la verdad": varían con la semilla y con el filtro. Entrenamos `k = 2…10` con **3 semillas** y mostramos la media y el rango. La coherencia **no se puede comparar entre corpus o idiomas como si fuese absoluta**: cada gráfica tiene su propia escala.

In [ ]:
KS = list(range(2, 11))
SEMILLAS = (0, 1, 2)
COH = {}
t0 = time.time()
for idioma, d in DATOS.items():
    cv = np.zeros((len(KS), len(SEMILLAS)))
    um = np.zeros((len(KS), len(SEMILLAS)))
    for i, k in enumerate(KS):
        for j, s in enumerate(SEMILLAS):
            m, dic, bow = entrenar_lda(d["base"], k, semilla=s)
            cv[i, j] = coherencia(m, d["base"], dic, bow, "c_v")
            um[i, j] = coherencia(m, d["base"], dic, bow, "u_mass")
    COH[idioma] = (cv, um)
    print("=====", idioma)
    tabla(["k", "c_v media", "c_v min", "c_v max", "u_mass media"],
          [[k, float(cv[i].mean()), float(cv[i].min()), float(cv[i].max()), float(um[i].mean())] for i, k in enumerate(KS)], ancho=16)
    print("k con mayor c_v medio:", KS[int(cv.mean(axis=1).argmax())], "| k con mayor u_mass medio:", KS[int(um.mean(axis=1).argmax())])
print("tiempo:", round(time.time() - t0), "s")

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for eje, (idioma, (cv, um)) in zip(ejes, COH.items()):
    eje.errorbar(KS, cv.mean(axis=1), yerr=[cv.mean(axis=1) - cv.min(axis=1), cv.max(axis=1) - cv.mean(axis=1)], marker="o", capsize=3)
    eje.set_title(f"c_v — {idioma}  (escala propia)")
    eje.set_xlabel("k")
    eje.set_ylabel("c_v")
plt.show()

### ¿Qué observas?

- La curva es **plana y ruidosa**: el rango entre semillas para un mismo `k` es del mismo orden que las diferencias entre `k` distintos. **No hay un `k` "correcto"** que la coherencia revele con claridad.
- c_v y u_mass **no tienen por qué coincidir** en el mejor `k` (compara las dos últimas líneas por idioma); u_mass empeora casi siempre al aumentar `k`, así que tiende a favorecer pocos temas.
- Criterio práctico: usa la coherencia para **descartar valores absurdos** y decide con criterio humano entre 2 o 3 candidatos, leyendo los temas (¿cada uno cuenta algo distinto?, ¿hay temas basura?). Y que un `k` tenga c_v más alto en español que en italiano **no dice nada** sobre qué idioma "se modela mejor": los corpus son distintos.

---
## 8. Experimento guiado: estabilidad con otras semillas y otros tamaños de documento

Entrenamos cinco veces con semillas 0-4 (`k = 5`) y medimos el parecido (Jaccard medio, sobre las 10 palabras principales) entre cada par de ejecuciones. Después repetimos con **pseudodocumentos más cortos o más largos** (100, 300 y 900 tokens) para ver el efecto del tamaño y del número de documentos.

In [ ]:
def estabilidad(docs, k=5, semillas=(0, 1, 2, 3, 4)):
    """Jaccard medio entre todos los pares de ejecuciones con semillas distintas."""
    tops = [temas_lda(entrenar_lda(docs, k, semilla=s, no_below=3)[0], 10) for s in semillas]
    pares = [jaccard_medio(tops[i], tops[j]) for i in range(len(tops)) for j in range(len(tops)) if i < j]
    return float(np.mean(pares)), float(np.min(pares)), float(np.max(pares))

for idioma, d in DATOS.items():
    print("=====", idioma)
    filas = []
    for n in (100, 300, 900):
        docs_n = [preparar(x, d["stop"], d["nombres"]) for x in trozos(d["tokens"], n)]
        media, minimo, maximo = estabilidad(docs_n)
        filas.append([n, len(docs_n), media, minimo, maximo])
    tabla(["tokens/doc", "documentos", "Jaccard medio", "min", "max"], filas, ancho=18)
    d["estab"] = filas

### ¿Qué observas?

Ninguna configuración da temas realmente estables: el Jaccard medio entre semillas queda lejos de 1 en todas. Con documentos muy cortos hay muchos documentos pero cada uno tiene pocas palabras; con documentos largos hay muy pocos. Con este corpus pequeño **no hay una combinación mágica**. Consecuencias prácticas: (1) fija la semilla y **documéntala**; (2) no presentes un solo entrenamiento como "los temas del libro"; (3) si un tema aparece en varias semillas, es más fiable que uno que aparece una vez.

---
## 9. ¿Recuperan LSA y LDA los dominios conocidos de las reseñas?

Las reseñas sintéticas tienen una columna `dominio` (`app`, `electronica`, `hotel`, `restaurante`, `servicio`, `tienda_online`: 6 valores). **`dominio` no es la etiqueta de sentimiento** (`etiqueta`): es el asunto de la reseña. Si un modelo de temas funciona, sus grupos deberían parecerse a los dominios. Para medirlo sin trampa:

1. Se ajusta el modelo **sin usar `dominio`** con `k = 6` (el número real de dominios: en un caso real no lo sabrías).
2. LSA no da grupos, así que se aplica **KMeans** sobre sus coordenadas normalizadas; en LDA se toma el **tema dominante**.
3. Se mide **pureza** (fracción de reseñas de la clase mayoritaria de su grupo) y **ARI** (*adjusted Rand index*: 1 = mismos grupos, ≈ 0 = como al azar, independiente de cómo se numeren los grupos).
4. Se comparan con una **asignación al azar** y con `etiqueta` (que no debería recuperarse).

Usamos el texto limpio (`texto_limpio`). Son 300 reseñas por idioma.

In [ ]:
ARCHIVOS = {"spa": "resenas_es.csv", "ita": "resenas_it.csv"}   # dato-es

def docs_resenas(idioma, agrupar=1):
    """Reseñas tokenizadas; con agrupar > 1 se juntan reseñas CONSECUTIVAS del mismo dominio (usa la columna dominio: control optimista)."""
    df = pd.read_csv(cargar_datos(ARCHIVOS[idioma]))
    stop = DATOS[idioma]["stop"]
    if agrupar == 1:
        textos, dominios, etiquetas = df.texto_limpio.tolist(), df.dominio.tolist(), df.etiqueta.tolist()
    else:
        textos, dominios, etiquetas = [], [], []
        for dom, g in df.sort_values(["dominio", "id"]).groupby("dominio"):
            t = g.texto_limpio.tolist()
            for i in range(0, len(t) - agrupar + 1, agrupar):
                textos.append(" ".join(t[i:i + agrupar]))
                dominios.append(dom)
                etiquetas.append("-")
    docs = [[w for w in re.findall(r"\w+", x.lower()) if w not in stop and len(w) >= 3] for x in textos]
    return docs, dominios, etiquetas

def evaluar(y, grupos):
    return round(adjusted_rand_score(y, grupos), 3), round(pureza(y, grupos), 3)

def lsa_lda_resenas(idioma, agrupar=1, k=6, semillas=(0, 1, 2)):
    docs, y, e = docs_resenas(idioma, agrupar)
    Z = TruncatedSVD(n_components=k, random_state=0).fit_transform(TfidfVectorizer(analyzer=lambda d: d, min_df=2, max_df=0.5).fit_transform(docs))
    g_lsa = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Normalizer().fit_transform(Z))
    res = {"n": len(docs), "LSA": evaluar(y, g_lsa), "LSA_etiqueta": evaluar(e, g_lsa)[0] if agrupar == 1 else None}
    lda = []
    for s in semillas:
        m, dic, bow = entrenar_lda(docs, k, semilla=s, no_below=2, no_above=0.5)
        lda.append(evaluar(y, mezcla(m, bow).argmax(axis=1)))
    res["LDA"] = lda
    res["azar"] = evaluar(y, np.random.RandomState(0).randint(0, k, len(docs)))
    return res

RES = {}
for idioma in ARCHIVOS:
    RES[idioma] = lsa_lda_resenas(idioma)
    r = RES[idioma]
    print(idioma, "reseñas:", r["n"], "| (ARI, pureza) | LSA+KMeans", r["LSA"], "| LDA (3 semillas)", r["LDA"], "| azar", r["azar"], "| ARI de LSA frente a etiqueta (sentimiento):", r["LSA_etiqueta"])

### ¿Qué observas?

Ni LSA ni LDA recuperan los dominios: el ARI es casi 0 (como el azar) y la pureza apenas supera la del azar. **No es un fallo del código**: las reseñas son muy cortas (≈ 9-10 palabras útiles), muchas no nombran ningún término propio de su dominio (`Cuidado, que luego pasa lo que pasa`) y **el vocabulario de opinión se mezcla entre dominios**. Ni siquiera un clasificador supervisado con TF-IDF pasa de una exactitud moderada al predecir `dominio` (lo comprobamos a continuación). Un modelo de temas solo puede recuperar lo que está en el vocabulario.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

for idioma in ARCHIVOS:
    docs, y, e = docs_resenas(idioma)
    X = TfidfVectorizer(analyzer=lambda d: d, min_df=2).fit_transform(docs)
    exactitud = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5).mean()
    print(idioma, "exactitud supervisada prediciendo dominio (CV 5):", round(float(exactitud), 3), "| azar:", round(1 / len(set(y)), 3))

### Control positivo (optimista): documentos más largos y homogéneos

Para ver si el método funciona cuando **el dato lo permite**, construimos pseudodocumentos juntando **10 reseñas consecutivas del mismo dominio** (27 documentos por idioma, unas 90-100 palabras útiles). **Ojo: esto usa `dominio` para construir los documentos**, así que el resultado es un **techo optimista**, no una evaluación honesta de un caso real.

In [ ]:
for idioma in ARCHIVOS:
    r = lsa_lda_resenas(idioma, agrupar=10)
    print(idioma, "documentos:", r["n"], "| (ARI, pureza) | LSA+KMeans", r["LSA"], "| LDA (3 semillas)", r["LDA"], "| azar", r["azar"])
    RES[idioma]["agrupado"] = r

### ¿Qué observas?

Con documentos largos y homogéneos, **LSA + KMeans recupera casi todos los dominios** (ARI cercano a 0,8-0,9), mientras que **LDA sigue siendo mucho peor** y cambia con la semilla: con solo 27 documentos, LDA (un modelo probabilístico con muchos parámetros) no tiene datos suficientes. Esto **no significa que LSA sea mejor que LDA en general**: significa que, en un corpus pequeño de textos cortos y con un `k` conocido, la solución determinista y más simple puede ser más robusta. Y recuerda: la comparación de ARI **entre español e italiano** solo dice cómo se comporta cada corpus sintético (son textos distintos), no qué idioma es más fácil.

---
## 10. Reproducibilidad

Con `random_state` fijo, dos entrenamientos con los mismos datos y parámetros dan **exactamente** los mismos temas. Lo comprobamos, y comprobamos también que **otra semilla** los cambia.

In [ ]:
d = DATOS["spa"]
m1 = entrenar_lda(d["base"], K, semilla=0)[0]
m2 = entrenar_lda(d["base"], K, semilla=0)[0]
m3 = entrenar_lda(d["base"], K, semilla=1)[0]
print("misma semilla, temas idénticos:", np.allclose(m1.get_topics(), m2.get_topics()))
print("otra semilla, temas idénticos:", np.allclose(m1.get_topics(), m3.get_topics()))
assert np.allclose(m1.get_topics(), m2.get_topics()) and not np.allclose(m1.get_topics(), m3.get_topics())

---
## Resumen

| Concepto | Idea clave |
|---|---|
| Tema y documento | tema = distribución sobre palabras; documento = mezcla de temas (`P(w\|d) = Σ θ·P(w\|t)`) |
| LSA | SVD truncada sobre TF-IDF; dimensiones con signo arbitrario, no son temas ni tienen etiqueta |
| LDA | modelo probabilístico entrenado sobre **recuentos**; `random_state` fijo; `k` lo eliges tú |
| Pre-tratamiento | nombres propios, stopwords, filtro de frecuencias y lemas **cambian los temas**; se declara cada decisión |
| Coherencia | c_v / u_mass: útil para descartar, ruidosa, **no comparable entre corpus ni idiomas** |
| Estabilidad | otra semilla, otro tamaño de documento, otros temas; corpus pequeño = temas poco fiables |
| Recuperar temas conocidos | se mide con pureza y ARI; con textos cortos no se recuperan; el control optimista usa la columna `dominio` |
| Etiquetas | las pone la persona, no el modelo |

**Siguiente:** en la clase 10 pasamos de las bolsas de palabras a **vectores densos** (*word embeddings*: Word2Vec y fastText). Después, la práctica de esta clase.